# Notebook 2 - Chunking Pipeline
### Rancang Bangun Chatbot Sumber Informasi Sivitas UPI Berbasis RAG

**Focus:** semantic chunking, metadata enrichment, chunk inspection &
retrieval-ready export.

**Pipeline position:** Stage B of 3. Consumes `cleaned/` from Notebook 1,
produces `chunked/chunks.jsonl` for `03_vectorstore_rag_evaluation.ipynb`.

**Outputs** (under `Dataset/_pipeline/chunked/`): `chunks.jsonl`,
`chunk_manifest.json`, `chunk_statistics.json`.

**Chunk metadata per record:** `chunk_id`, `chunk_index`, `doc_id`, `source`,
`url`, `title`, `category`, `source_type`, `page`, `section`, `text`,
`chunk_length`.

## Cell 1 - Install dependencies

In [ ]:
# =============================================================================
# Notebook 2 - Install dependencies (run once per Colab session)
# =============================================================================
!pip install -q \
    langchain-text-splitters==0.3.2 \
    pandas==2.2.2 \
    tqdm==4.66.5

print("\nInstall step complete.")

## Cell 2 - Mount Drive, imports, config

*Always run first.* Includes a dependency guard that checks Notebook 1's `cleaned/` output exists.

In [ ]:
# =============================================================================
# Notebook 2 - Imports, environment detection, logging, config  (ALWAYS run first)
# =============================================================================
import json
import re
import os
import logging
import statistics
from pathlib import Path
from datetime import datetime

import pandas as pd
from tqdm.auto import tqdm

# --- Environment detection: Colab vs local (Windows-friendly) ---
IN_COLAB = False
try:
    from google.colab import drive
    drive.mount("/content/drive")
    IN_COLAB = True
except Exception:
    IN_COLAB = False
    print("Local mode (no Colab). Using local Windows-safe paths.")


def get_logger(name="nb2", logfile=None):
    logger = logging.getLogger(name)
    logger.setLevel(logging.INFO)
    fmt = logging.Formatter("%(asctime)s | %(levelname)-7s | %(message)s",
                            datefmt="%H:%M:%S")
    if not any(isinstance(h, logging.StreamHandler) for h in logger.handlers):
        sh = logging.StreamHandler(); sh.setFormatter(fmt); logger.addHandler(sh)
    if logfile and not any(isinstance(h, logging.FileHandler) for h in logger.handlers):
        try:
            fh = logging.FileHandler(logfile, encoding="utf-8")
            fh.setFormatter(fmt); logger.addHandler(fh)
        except Exception:
            pass
    logger.propagate = False
    return logger


# --------------------------------------------------------------------------
# CONFIG - edit here only.
# Base path resolution: env var > Colab > local Windows default.
# --------------------------------------------------------------------------
_env_base = os.environ.get("RAG_UPI_BASE")
if _env_base:
    DRIVE_BASE = Path(_env_base)
elif IN_COLAB:
    DRIVE_BASE = Path("/content/drive/MyDrive/Dataset")
else:
    DRIVE_BASE = Path(r"D:\Project\RAG_UPI\Dataset")

CONFIG = {
    "pipeline_dir": DRIVE_BASE / "_pipeline",
    # Chunking parameters (characters). Tune for your corpus.
    "chunk_size": 1000,
    "chunk_overlap": 200,
    "min_chunk_chars": 50,        # discard fragments shorter than this
}

PIPE = CONFIG["pipeline_dir"]
DIRS = {
    "cleaned": PIPE / "cleaned",   # INPUT  (produced by Notebook 1)
    "chunked": PIPE / "chunked",   # OUTPUT (consumed by Notebook 3)
    "logs":    PIPE / "logs",
}
for d in (DIRS["chunked"], DIRS["logs"]):
    d.mkdir(parents=True, exist_ok=True)

log = get_logger("nb2", logfile=str(DIRS["logs"] / "nb2_chunking.log"))


def read_json(path, default=None):
    path = Path(path)
    if not path.exists():
        return default
    try:
        return json.loads(path.read_text(encoding="utf-8"))
    except json.JSONDecodeError as e:
        log.warning("Corrupt JSON %s (%s).", path, e)
        return default


def write_json(path, obj):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(json.dumps(obj, ensure_ascii=False, indent=2), encoding="utf-8")
    tmp.replace(path)


# --- Dependency guard: Notebook 2 needs Notebook 1's output ---
if not DIRS["cleaned"].exists():
    log.error("Cleaned folder missing: %s", DIRS["cleaned"])
    log.error("Run Notebook 1 (01_cleaning_extraction_pipeline.ipynb) first.")
else:
    n = len([f for f in DIRS["cleaned"].glob("*.json")
             if f.name != "cleaned_manifest.json"])
    log.info("Config loaded. Base dir: %s", DRIVE_BASE)
    log.info("Found %d cleaned documents to chunk.", n)


## Cell 3 - Semantic chunking + metadata enrichment

Recursive splitter with heading-aware separators (`\n## ` is highest priority), configurable `chunk_size`/`chunk_overlap`. Each chunk is tagged with the nearest preceding `## HEADING` as its `section`. The JSONL is rebuilt fresh each run so it always matches the current cleaned corpus.

In [ ]:
# =============================================================================
# Notebook 2 - Stage: Semantic chunking + metadata enrichment
# =============================================================================
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Heading markers produced by Notebook 1's cleaner ("## HEADING").
HEADING_MARK = re.compile(r"^##\s+(.+)$", re.M)


def make_splitter():
    """Recursive splitter: respects paragraph/sentence boundaries first.

    Separator priority means it breaks on blank lines, then newlines, then
    sentence punctuation, before resorting to mid-word splits. This keeps
    headings and bullet lists intact within a chunk where possible.
    """
    return RecursiveCharacterTextSplitter(
        chunk_size=CONFIG["chunk_size"],
        chunk_overlap=CONFIG["chunk_overlap"],
        separators=["\n## ", "\n\n", "\n", ". ", "? ", "! ", "; ", " ", ""],
        length_function=len,
        keep_separator=True,
    )


def _nearest_heading(page_text, upto_offset):
    """Return the most recent '## HEADING' before a character offset, if any."""
    head = None
    for m in HEADING_MARK.finditer(page_text):
        if m.start() <= upto_offset:
            head = m.group(1).strip()
        else:
            break
    return head


def chunk_document(doc, splitter):
    """Chunk one cleaned document into enriched records.

    Each chunk carries metadata: source, url, title, category, page,
    chunk_index, source_type, chunk_length, section heading.
    """
    records = []
    for page in doc["pages"]:
        text = page.get("text", "")
        if len(text.strip()) < CONFIG["min_chunk_chars"]:
            continue
        offset = 0
        for piece in splitter.split_text(text):
            piece_stripped = piece.strip()
            if len(piece_stripped) < CONFIG["min_chunk_chars"]:
                offset += len(piece)
                continue
            heading = _nearest_heading(text, offset)
            records.append({
                "doc_id": doc["id"],
                "source": doc["source"],
                "url": doc.get("url"),
                "title": doc["title"],
                "category": doc["category"],
                "source_type": doc.get("source_type", "unknown"),
                "page": page["page"],
                "section": heading,
                "text": piece_stripped,
                "chunk_length": len(piece_stripped),
            })
            offset += len(piece)
    return records


def run_chunking():
    """Chunk every cleaned document into a single chunks.jsonl with metadata.

    The JSONL is rebuilt fresh each run so it always matches the current
    cleaned corpus (no stale or duplicated chunks). A global chunk_index is
    assigned in deterministic order.
    """
    cleaned = sorted(f for f in DIRS["cleaned"].glob("*.json")
                     if f.name != "cleaned_manifest.json")
    if not cleaned:
        log.error("No cleaned documents found. Run Notebook 1 first.")
        return

    splitter = make_splitter()
    out_path = DIRS["chunked"] / "chunks.jsonl"
    manifest_rows, global_idx = [], 0

    with out_path.open("w", encoding="utf-8") as fout:
        for f in tqdm(cleaned, desc="Chunking", unit="doc"):
            doc = read_json(f)
            if not doc:
                continue
            recs = chunk_document(doc, splitter)
            for r in recs:
                r["chunk_index"] = global_idx
                r["chunk_id"] = f"{r['doc_id']}::{global_idx}"
                fout.write(json.dumps(r, ensure_ascii=False) + "\n")
                global_idx += 1
            manifest_rows.append({
                "doc_id": doc["id"], "title": doc["title"],
                "category": doc["category"],
                "source_type": doc.get("source_type", "unknown"),
                "n_chunks": len(recs),
            })

    write_json(DIRS["chunked"] / "chunk_manifest.json", manifest_rows)
    log.info("Chunking complete: %d chunks from %d docs -> %s",
             global_idx, len(manifest_rows), out_path)
    return out_path


# ---- RUN CHUNKING ----
chunks_path = run_chunking()

## Cell 4 - SELF-REVIEW + chunk statistics + TEST SUITE

**Chunking tradeoffs (the central design decision)**
- *Small chunks* (~300-500 chars): precise retrieval, but a chunk may lack the
  context needed to answer; more chunks => larger index, slower eval.
- *Large chunks* (~1500+ chars): rich context, but retrieval precision drops
  (one chunk mixes several topics) and the LLM context fills faster.
- **Chosen default 1000/200** is a balanced starting point for UPI documents
  (mixed regulations, circulars, news, facility pages). It is `CONFIG`-driven:
  re-run this notebook with different values and compare retrieval metrics in
  Notebook 3 - that sweep is itself a defensible thesis methodology section.

**Architectural notes**
- Heading-aware separators keep regulation clauses and list items intact where
  possible, but a chunk boundary can still split a long bullet list - acceptable
  given the overlap.
- `section` is best-effort: it depends on Notebook 1 having marked headings.
  Documents with no detectable headings simply get `section = null`.

**Failure points & handling**
- Missing/empty `cleaned/` -> dependency guard logs an explicit instruction.
- Corrupt cleaned JSON -> `read_json` logs and skips that document.
- A page shorter than `min_chunk_chars` -> skipped (not an error).

**Runtime & memory risks**
- Chunking is fast and CPU-light; documents stream one at a time. The JSONL is
  written incrementally, so memory stays flat even for large corpora.

**Test suite covers:** chunk schema completeness, globally unique + contiguous
`chunk_index`, size bounds, manifest/metadata consistency, and a spot-check that
consecutive chunks genuinely overlap.

## Cell 4 (code) - statistics + tests + inspection

In [ ]:
# =============================================================================
# Notebook 2 - Chunk statistics, inspection & TEST SUITE
# =============================================================================

def load_chunks():
    """Load chunks.jsonl into a list of dicts."""
    p = DIRS["chunked"] / "chunks.jsonl"
    if not p.exists():
        log.error("chunks.jsonl not found. Run the chunking cell first.")
        return []
    return [json.loads(ln) for ln in p.read_text(encoding="utf-8").splitlines()
            if ln.strip()]


def chunk_statistics():
    """Compute and save corpus-level chunk statistics; return a DataFrame."""
    chunks = load_chunks()
    if not chunks:
        return None
    df = pd.DataFrame(chunks)
    lengths = df["chunk_length"].tolist()
    stats = {
        "total_chunks": len(df),
        "total_documents": df["doc_id"].nunique(),
        "chunk_length_min": int(min(lengths)),
        "chunk_length_max": int(max(lengths)),
        "chunk_length_mean": round(statistics.mean(lengths), 1),
        "chunk_length_median": int(statistics.median(lengths)),
        "by_category": df.groupby("category").size().to_dict(),
        "by_source_type": df.groupby("source_type").size().to_dict(),
        "chunks_with_section": int(df["section"].notna().sum()),
        "generated_at": datetime.now().isoformat(timespec="seconds"),
    }
    write_json(DIRS["chunked"] / "chunk_statistics.json", stats)
    log.info("Chunk statistics saved. Summary:")
    for k, v in stats.items():
        log.info("  %s: %s", k, v)
    return df


def inspect_chunks(n=5, category=None, source_type=None):
    """Print a sample of chunks, optionally filtered by category/source_type."""
    chunks = load_chunks()
    if category:
        chunks = [c for c in chunks if c["category"] == category]
    if source_type:
        chunks = [c for c in chunks if c["source_type"] == source_type]
    if not chunks:
        print("No chunks match the filter.")
        return
    for c in chunks[:n]:
        print(f"\n[chunk {c['chunk_index']}] doc={c['doc_id']} "
              f"page={c['page']} len={c['chunk_length']}")
        print(f"  title  : {c['title']}")
        print(f"  section: {c.get('section')}")
        print(f"  text   : {c['text'][:300]}...")


# --- TEST SUITE ---
def _check(name, condition, detail=""):
    tag = "PASS" if condition else "FAIL"
    log.info("[%s] %s %s", tag, name, f"- {detail}" if detail else "")
    return bool(condition)


def test_chunk_schema():
    """Every chunk must carry the full required metadata set."""
    chunks = load_chunks()
    if not chunks:
        return _check("chunks exist", False)
    required = {"chunk_id", "chunk_index", "doc_id", "source", "title",
                "category", "source_type", "page", "text", "chunk_length"}
    ok = True
    for c in chunks[:200]:
        missing = required - set(c.keys())
        ok &= _check(f"chunk {c.get('chunk_index')}: schema",
                     not missing, f"missing {missing}" if missing else "")
        if not ok:
            break
    return ok


def test_chunk_indices_unique():
    """Global chunk_index values must be unique and contiguous from 0."""
    chunks = load_chunks()
    if not chunks:
        return False
    idxs = sorted(c["chunk_index"] for c in chunks)
    ok = _check("chunk_index unique", len(idxs) == len(set(idxs)))
    ok &= _check("chunk_index contiguous from 0",
                 idxs == list(range(len(idxs))))
    return ok


def test_chunk_sizes():
    """Chunk lengths must respect min size and not wildly exceed chunk_size."""
    chunks = load_chunks()
    if not chunks:
        return False
    too_small = [c for c in chunks if c["chunk_length"] < CONFIG["min_chunk_chars"]]
    # Overlap + separator retention can exceed chunk_size slightly; allow 1.5x.
    too_big = [c for c in chunks
               if c["chunk_length"] > CONFIG["chunk_size"] * 1.5]
    ok = _check("no undersized chunks", not too_small, f"{len(too_small)} found")
    ok &= _check("no grossly oversized chunks", not too_big,
                 f"{len(too_big)} found")
    return ok


def test_metadata_consistency():
    """Chunk doc_ids must all appear in the chunk manifest."""
    chunks = load_chunks()
    manifest = read_json(DIRS["chunked"] / "chunk_manifest.json", default=[])
    man_ids = {r["doc_id"] for r in manifest}
    chunk_ids = {c["doc_id"] for c in chunks}
    return _check("all chunk doc_ids in manifest",
                  chunk_ids <= man_ids,
                  f"{len(chunk_ids - man_ids)} orphan doc_ids")


def test_overlap_present():
    """Spot-check that consecutive same-page chunks actually overlap."""
    chunks = load_chunks()
    if CONFIG["chunk_overlap"] == 0:
        return _check("overlap configured off", True)
    by_doc_page = {}
    for c in chunks:
        by_doc_page.setdefault((c["doc_id"], c["page"]), []).append(c)
    checked = overlapped = 0
    for seq in by_doc_page.values():
        seq.sort(key=lambda c: c["chunk_index"])
        for a, b in zip(seq, seq[1:]):
            checked += 1
            tail = a["text"][-CONFIG["chunk_overlap"]:]
            # Some token sequence of a's tail should reappear in b.
            if any(w and w in b["text"] for w in tail.split()[:8]):
                overlapped += 1
    ratio = overlapped / checked if checked else 1.0
    return _check("consecutive chunks overlap", ratio >= 0.5,
                  f"{overlapped}/{checked} pairs overlap")


def run_all_tests():
    log.info("=" * 60)
    log.info("NOTEBOOK 2 - TEST SUITE")
    log.info("=" * 60)
    results = {
        "schema": test_chunk_schema(),
        "indices_unique": test_chunk_indices_unique(),
        "sizes": test_chunk_sizes(),
        "metadata": test_metadata_consistency(),
        "overlap": test_overlap_present(),
    }
    passed = sum(1 for v in results.values() if v)
    log.info("=" * 60)
    log.info("RESULT: %d/%d test groups passed. %s",
             passed, len(results), results)
    return results


# ---- RUN STATS + TESTS ----
stats_df = chunk_statistics()
test_results = run_all_tests()
print("\nTip: inspect_chunks(5, category='PPID') to preview chunks by category.")

---
### Re-running (Notebook 2)
Run Cell 2 first, then `run_chunking()` / `chunk_statistics()` / `run_all_tests()`.
Inspect with `inspect_chunks(5, category='PPID', source_type='pdf')`.

**Next:** open `03_vectorstore_rag_evaluation.ipynb`.